# 01 — Integrasi, Audit & Cleaning (v3: generik, tahan data baru)

**Tujuan:** membaca semua file mentah di `data/raw/`, **membuktikan dengan angka** masalah di dalamnya, lalu membangun tabel bersih yang aman dipakai notebook 02–05.

**Beda utama v3 dibanding v2** (isi tabel temuan v2 tetap berlaku; yang berubah adalah cara menemukannya):
1. **Tidak ada lagi tahun/kabupaten/kode yang di-hardcode.** Cacat ditemukan oleh *detektor generik* (skala ×10ⁿ, tahun tak lengkap, salinan antar tahun, lonjakan baris), bukan "tahun 2021 pasti ×1000".
2. **Nama file fleksibel & boleh banyak file per jenis** (mis. `...2020-2025.xlsx` lalu `...2020-2026.xlsx`, atau file tambahan hanya untuk tahun baru). Untuk tahun yang sama, file terbaru menang.
3. **Luas lahan TIDAK dibuang.** Baris/tahun yang rusak ditandai `NaN` + alasan, sisanya dipakai; sumber KKP (file ke-10) dipakai bila tersedia.
4. **Tidak ada `assert` yang mematikan alur.** Masalah masuk `qc_report.csv` (INFO/WARN/ERROR). Alur hanya berhenti bila *tidak ada data yang bisa dipakai sama sekali*.

In [1]:
import os, re, json, hashlib, warnings, unicodedata
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

RAW  = Path('../data/raw')
PROC = Path('../data/processed'); PROC.mkdir(parents=True, exist_ok=True)
REP  = Path('../output/reports');  REP.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 200); pd.set_option('display.max_rows', 120)

QC, LOG, YEAR_AUDIT = [], [], []          # catatan kualitas / keputusan cleaning / hasil audit tahun
def qc(level, sumber, pesan, tahun=None):
    QC.append(dict(level=level, sumber=sumber, tahun=tahun, pesan=pesan)); print(f'[{level}] {sumber}: {pesan}')
def log(sumber, masalah, bukti, tindakan):
    LOG.append(dict(sumber=sumber, masalah=masalah, bukti=bukti, tindakan=tindakan))

## 1. Helper

In [2]:
def parse_number(x):
    '''Angka dari excel/teks. '-' dan sejenisnya -> NaN. (*) dibuang. Mendukung "1,234.5" dan "1.234,5".'''
    if x is None or (not isinstance(x, str) and pd.isna(x)): return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)): return float(x)
    s = re.sub(r'\(\*+\)', '', str(x)).strip()
    if s in {'-', '–', '—', '', 'nan', 'NaN', 'NA', 'n/a', '..', '...'}: return np.nan
    if ',' in s and '.' in s: s = s.replace('.', '').replace(',', '.') if s.rfind(',') > s.rfind('.') else s.replace(',', '')
    elif ',' in s: s = s.replace(',', '') if re.fullmatch(r'-?\d{1,3}(,\d{3})+', s) else s.replace(',', '.')
    try: return float(s)
    except ValueError: return np.nan

def norm_txt(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+', ' ', s).strip()

def norm_kab(s):
    if s is None or (not isinstance(s, str) and pd.isna(s)): return s
    s = re.sub(r'\s+', ' ', str(s).upper().strip())
    s = re.sub(r'^(KABUPATEN|KAB\.?)\s+', '', s)
    s = re.sub(r'^(KOTAMADYA|KOTA)\s+', 'KOTA ', s)
    return s.title()

def pick(df, *pats, required=True):
    '''Cari kolom lewat pola regex pada nama kolom yang sudah dinormalkan. Pola yang lebih awal = prioritas.'''
    cols = {c: norm_txt(c) for c in df.columns if not str(c).startswith('_')}
    for pat in pats:
        for c, n in cols.items():
            if re.search(pat, n): return c
    if required: raise KeyError(f'kolom {pats} tidak ditemukan. Kolom yang ada: {list(cols.values())}')

SYN_KOM  = {'GURAME; GURAMI': 'GURAMI', 'GURAME': 'GURAMI'}
GRUP_KOM = {'RUMPUT LAUT': 'rumput_laut', 'UDANG': 'udang', 'UDANG LAINNYA': 'udang', 'UDANG VANAME': 'udang', 'UDANG WINDU': 'udang',
            'BANDENG': 'bandeng', 'LELE': 'lele', 'NILA': 'nila', 'GURAMI': 'gurami', 'PATIN': 'patin', 'MAS': 'mas'}

## 2. Penemuan file otomatis
Nama file boleh berubah; yang dicari hanya **kata kunci**. File `.bak`, `~$...`, dan folder tersembunyi diabaikan.
Jika ada beberapa file untuk satu jenis, semuanya dibaca; untuk tahun yang tercakup file lebih baru, file lama *tidak dipakai*.

In [3]:
RULES = {   # kunci: (kata WAJIB ada di nama file, kata yang TIDAK BOLEH ada)
 'produksi_jenis': (['jumlah', 'produksi', 'jenis budidaya'], ['kegiatan', 'pembenihan', 'pembesaran', 'hias']),
 'rtp':            (['rtp'], []),
 'luas':           (['luas bersih'], []),
 'luas_kkp':       (['luas', 'lahan'], ['bersih']),            # file ke-10: Luas Lahan Budidaya (KKP) - opsional
 'pembudidaya':    (['pembudidaya'], []),
 'hias':           (['ikan hias'], []),
 'bps_kegiatan':   (['jenis kegiatan'], []),
 'pembenihan':     (['pembenihan'], ['jenis kegiatan', 'ikan hias']),
 'pembesaran':     (['pembesaran'], ['jenis kegiatan']),
 'bps_nilai':      (['nilai produksi'], []),
}
REQUIRED = ['produksi_jenis', 'pembesaran', 'pembudidaya']       # tanpa ini panel tidak bisa dibangun

def all_files():
    return [p for p in RAW.rglob('*') if p.is_file() and p.suffix.lower() in ('.xlsx', '.xls', '.csv')
            and not p.name.startswith(('~$', '.')) and not any(part.startswith('.') for part in p.relative_to(RAW).parts)]

def find_files(key):
    must, never = RULES[key]
    hit = [p for p in all_files() if all(norm_txt(m) in norm_txt(p.stem) for m in must) and not any(norm_txt(n) in norm_txt(p.stem) for n in never)]
    return sorted(hit, key=lambda p: (p.stat().st_mtime, p.name))          # tertua -> terbaru

def read_any(p):
    if p.suffix.lower() == '.csv':
        try: return pd.read_csv(p, sep=None, engine='python', encoding='utf-8-sig')
        except UnicodeDecodeError: return pd.read_csv(p, sep=None, engine='python', encoding='latin-1')
    return pd.read_excel(p)

FILES_USED = {}
def read(key):
    fs = find_files(key); FILES_USED[key] = [p.name for p in fs]
    if not fs:
        qc('ERROR' if key in REQUIRED else 'INFO', key, 'file tidak ditemukan di data/raw' + (' (WAJIB)' if key in REQUIRED else ' (opsional, dilewati)')); return None
    parts = []
    for p in fs:
        d = read_any(p); d.columns = [str(c).strip() for c in d.columns]; d['_file'] = p.name; parts.append(d)
    if len(fs) > 1: qc('INFO', key, f'{len(fs)} file digabung; tahun yang sama -> file terbaru menang: ' + ', '.join(p.name for p in fs))
    return pd.concat(parts, ignore_index=True)

def newest_per_year(df, ycol='tahun'):
    '''Untuk tahun yang ada di beberapa file, pakai hanya file TERBARU (urutan file = tertua -> terbaru).'''
    if df['_file'].nunique() < 2: return df
    claimed, keep = set(), []
    for f in list(dict.fromkeys(df['_file']))[::-1]:
        sub = df[df['_file'] == f]; ys = set(sub[ycol].dropna())
        keep.append(sub[~sub[ycol].isin(claimed)]); claimed |= ys
    return pd.concat(keep).sort_index()

print('File terdeteksi:'); [print(f'  {k:15s}', v) for k, v in {k: [p.name for p in find_files(k)] for k in RULES}.items()]

File terdeteksi:
  produksi_jenis  ['Jumlah Produksi Perikanan Budidaya Berdasarkan Jenis Budidaya 2020-2025.xlsx']
  rtp             ['Jumlah RTP (Rumah Tangga Perikanan) 2020-2024.xlsx']
  luas            ['Luas Bersih Lahan Perikanan Budidaya Berdasarkan Jenis Budidaya 2020-2024.xlsx']
  luas_kkp        ['Luas lahan Budidaya 2020-2024.xlsx']
  pembudidaya     ['Pembudidaya Ikan 2020-2024.xlsx']
  hias            ['Produksi Perikanan Budidaya Ikan Hias 2020-2024.xlsx']
  bps_kegiatan    ['Produksi Perikanan Budidaya Menurut KabupatenKota dan Jenis Kegiatan di Provinsi Jawa Timur 2020-2022.xlsx']
  pembenihan      ['Produksi Perikanan Budidaya Pembenihan 2020-2024.xlsx']
  pembesaran      ['Produksi Perikanan Budidaya Pembesaran 2020-2024.xlsx']
  bps_nilai       ['Volume Produksi dan Nilai Produksi Perikanan Budidaya Menurut KabupatenKota dan Jenis Budidaya di Provinsi Jawa Timur 2023-2024.xlsx']


[None, None, None, None, None, None, None, None, None, None]

## 3. Detektor generik (inti v3)
Semua pemeriksaan bekerja pada matriks **kabupaten × tahun** dan *membandingkan tiap tahun dengan tahun-tahun lain*.

| status | arti | tindakan |
|---|---|---|
| `skala` | seluruh tahun berselisih 10³ / 10⁴ / 10⁶ secara **seragam** di semua kabupaten (salah satuan / pemisah desimal hilang) | dikalikan 10⁻ⁿ (dan dicatat) |
| `tidak_lengkap` | ≥80% kabupaten nol/kosong padahal tahun lain terisi (nol = belum lapor) | tahun → `NaN` |
| `tidak_konsisten` | menyimpang dari tahun lain tetapi **bukan** kelipatan skala yang seragam (tercampur data lain) | tahun → `NaN` |
| `salinan` | ≥90% kabupaten bernilai sama persis dengan tahun lain, padahal antar-tahun normal jarang sama | tahun terbaru → `NaN` |
| `sebagian` | 20–80% kabupaten kosong tetapi sisanya wajar | dipakai; dashboard menyaring lewat gate cakupan |
| `tak_terverifikasi` | kurang dari 2 tahun pembanding | dipakai apa adanya, diberi catatan |

In [4]:
ALLOWED_K = (3, 4, 6)     # 10^±3 (x1000), 10^±4 (m2<->ha), 10^±6

def audit_years(W, min_ref_years=2, min_kab=5, lenient=False):
    W = W.copy(); W.columns = [int(c) for c in W.columns]; years = sorted(W.columns)
    res = {y: dict(tahun=y, status='ok', faktor=1.0, median_log10=np.nan, iqr_log10=np.nan, zero_share=np.nan, n_kab_ada=int((W[y] > 0).sum())) for y in years}
    bad = set()
    for _ in range(4):
        new_bad = set()
        for y in years:
            r = res[y]; r.update(status='ok', faktor=1.0, median_log10=np.nan, iqr_log10=np.nan, zero_share=np.nan)
            oth = [z for z in years if z != y and z not in bad]
            if len(oth) < min_ref_years: r['status'] = 'tak_terverifikasi'; continue
            ref = W[oth].where(W[oth] > 0).median(axis=1); has = ref.notna()
            if has.sum() < min_kab: r['status'] = 'tak_terverifikasi'; continue
            v = W.loc[has, y].fillna(0); zs = float((v <= 0).mean()); r['zero_share'] = zs
            if zs >= 0.8: r['status'] = 'tidak_lengkap'; new_bad.add(y); continue
            lr = np.log10(v[v > 0] / ref[has][v > 0]); m = float(lr.median()); iq = float(lr.quantile(.75) - lr.quantile(.25)); k = int(round(m))
            r['median_log10'], r['iqr_log10'] = m, iq
            if abs(m) < 0.35: r['status'] = 'sebagian' if zs > 0.2 else 'ok'
            elif abs(k) in ALLOWED_K and abs(m - k) < 0.25 and iq < 0.6: r['status'] = 'skala'; r['faktor'] = 10.0 ** (-k); new_bad.add(y)
            elif lenient: r['status'] = 'sebagian'           # data jarang (hanya kabupaten aktif): jangan terlalu galak
            else: r['status'] = 'tidak_konsisten'; new_bad.add(y)
        if new_bad == bad: break
        bad = new_bad
    return pd.DataFrame(list(res.values())).set_index('tahun')

def stale_years(W, thr=0.9, min_n=8, base_max=0.5):
    '''Tahun y yang >=thr nilainya identik dengan tahun lebih awal z, padahal pasangan tahun lain jarang identik.'''
    W = W.copy(); W.columns = [int(c) for c in W.columns]; ys = sorted(W.columns)
    def same(a, b):
        m = (W[a] > 0) & (W[b] > 0)
        return float(np.isclose(W.loc[m, a], W.loc[m, b]).mean()) if m.sum() >= min_n else np.nan
    adj = [same(ys[i - 1], ys[i]) for i in range(1, len(ys))]; out = {}
    for i, y in enumerate(ys):
        for z in ys[:i]:
            s = same(z, y)
            if pd.notna(s) and s >= thr:
                others = [a for a in adj if pd.notna(a) and a < thr]
                if others and np.median(others) < base_max: out[y] = z; break
    return out

def drop_spikes(df, keys, ycol='tahun', vcol='nilai', ratio=50, glob_ratio=200, max_share=0.10):
    '''Baris yang melonjak >ratio x median baris yang sama di tahun lain (atau >glob_ratio x median jenis tsb di tahun itu).
       Hanya dilakukan bila lonjakan <=10% baris tahun itu; bila lebih banyak, itu masalah TAHUN (ditangani audit_years), bukan baris.'''
    d = df.copy(); flag = pd.Series(False, index=d.index)
    for y in d[ycol].dropna().unique():
        oth = d[(d[ycol] != y) & (d[vcol] > 0)].groupby(keys)[vcol].median().rename('_ref')
        cur = d[(d[ycol] == y) & (d[vcol] > 0)].join(oth, on=keys)
        gm = cur.groupby(keys[-1])[vcol].transform('median')
        bad = ((cur[vcol] > ratio * cur['_ref']) | (cur[vcol] > glob_ratio * gm)) & (cur[vcol] > 100)
        if len(cur) and bad.sum() / len(cur) <= max_share: flag[bad.index[bad]] = True
    d.loc[flag, vcol] = np.nan
    return d, int(flag.sum())

def apply_audit(df, name, kcol='kabupaten', ycol='tahun', vcol='nilai', stale=False, lenient=False):
    '''Jalankan audit tahun pada total kabupaten x tahun, lalu koreksi baris di df. Mengembalikan (df_baru, tabel_audit).'''
    d = df.copy(); W = d.groupby([kcol, ycol])[vcol].sum(min_count=1).unstack(ycol)
    if W.shape[1] == 0 or W.dropna(how='all').empty: return d, pd.DataFrame()
    aud = audit_years(W, lenient=lenient); st = stale_years(W) if stale else {}
    for y, r in aud.iterrows():
        m = d[ycol] == y
        if y in st: aud.loc[y, 'status'] = 'salinan'; d.loc[m, vcol] = np.nan; qc('WARN', name, f'tahun {y} salinan tahun {st[y]} (>=90% kabupaten identik) -> NaN', y)
        elif r.status == 'skala': d.loc[m, vcol] *= r.faktor; qc('WARN', name, f'tahun {y} berskala x{1/r.faktor:g} -> dikoreksi', y); log(name, f'tahun {y} berskala x{1/r.faktor:g}', f'median log10 rasio={r.median_log10:.2f}, IQR={r.iqr_log10:.2f} (seragam)', f'dikalikan {r.faktor:g}')
        elif r.status in ('tidak_lengkap', 'tidak_konsisten'):
            d.loc[m, vcol] = np.nan; qc('WARN', name, f'tahun {y} {r.status} (zero_share={r.zero_share:.0%}, median_log10={r.median_log10:.2f}) -> NaN', y)
            log(name, f'tahun {y} {r.status}', f'zero_share={r.zero_share:.0%}; median log10 rasio ke tahun lain={r.median_log10:.2f}; IQR={r.iqr_log10:.2f}', 'tahun dijadikan NaN (bukan 0)')
        elif r.status == 'tak_terverifikasi': qc('INFO', name, f'tahun {y} tak terverifikasi (pembanding <2 tahun)', y)
    a = aud.reset_index(); a.insert(0, 'dataset', name); YEAR_AUDIT.append(a)
    return d, aud

## 4. Opendata: produksi per jenis, RTP, luas  (+ pemisahan Kabupaten/Kota kembar secara generik)

In [5]:
def load_opendata(key):
    d = read(key)
    if d is None: return None
    kc = pick(d, r'^kode kabupaten kota$', r'kode kab', required=False)
    nc = pick(d, r'^nama kabupaten kota$', r'nama kab', r'kabupaten|kota')
    vc = pick(d, r'^jumlah$', r'^nilai$', r'jumlah'); tc = pick(d, r'^tahun$'); gc = pick(d, r'^kategori$', r'jenis')
    o = pd.DataFrame({'tahun': pd.to_numeric(d[tc], errors='coerce'), 'kode': d[kc].astype(str).str.replace(r'\.0$', '', regex=True) if kc else '', 'nama_raw': d[nc].astype(str),
                      'jenis': d[gc].astype(str).str.upper().str.strip(), 'nilai': d[vc].map(parse_number), '_file': d['_file']}).dropna(subset=['tahun'])
    o['tahun'] = o.tahun.astype(int); o = newest_per_year(o)
    o['agregat'] = o.kode.isin(['0', '', 'nan']) | o.nama_raw.map(norm_txt).isin(['jawa timur', 'provinsi jawa timur', 'prov jawa timur'])
    o['blok'] = o.groupby(['tahun', 'kode', 'jenis']).cumcount()
    return o

def resolve_twins(o, name):
    '''Kode kembar Kab/Kota = kode bernama "KOTA x" yang SETIAP (tahun,jenis)-nya muncul 2x (>=90%). Kemunculan pertama = KABUPATEN x, kedua = KOTA x (dibuktikan di v2 dgn BPS).
       Baris ke-3 dst pada kunci yang sama (unggah ganda / salin-tempel) dibuang; yang dipakai baris pertama.'''
    o = o.copy(); nonag = o[~o.agregat]
    cnt = nonag.groupby(['kode', 'tahun', 'jenis']).size(); share2 = (cnt >= 2).groupby(level='kode').mean()
    twin = set(share2[share2 >= 0.9].index) & set(nonag[nonag.nama_raw.str.upper().str.startswith('KOTA')].kode)
    extra = (o.blok >= np.where(o.kode.isin(twin), 2, 1)) & ~o.agregat
    if extra.any(): qc('WARN', name, f'{int(extra.sum())} baris ganda (kunci tahun-kode-jenis sama) dibuang; baris pertama dipakai'); o = o[~extra].copy()
    o['kabupaten'] = o.nama_raw.map(norm_kab)
    m = o.kode.isin(twin) & (o.blok == 0) & ~o.agregat
    o.loc[m, 'kabupaten'] = o.loc[m, 'nama_raw'].str.upper().str.replace(r'^KOTA\s+', '', regex=True).str.title()
    if twin: qc('INFO', name, f'{len(twin)} kode kembar Kab/Kota dipisahkan (kemunculan pertama = Kabupaten): ' + ', '.join(sorted(o.loc[m, 'kabupaten'].unique())))
    return o

prod_raw = load_opendata('produksi_jenis'); rtp_raw = load_opendata('rtp'); luas_raw = load_opendata('luas')
if prod_raw is None: raise SystemExit('Berhenti: file produksi jenis budidaya WAJIB ada.')
prod_od = resolve_twins(prod_raw, 'produksi_jenis'); prod_od = prod_od[~prod_od.agregat]
rtp_od  = resolve_twins(rtp_raw,  'rtp')  if rtp_raw  is not None else None
luas_od = resolve_twins(luas_raw, 'luas') if luas_raw is not None else None
for nm, d in [('produksi', prod_od), ('rtp', rtp_od), ('luas', luas_od)]:
    if d is not None: print(f'{nm:9s} baris={len(d):5d} | kab unik={d[~d.agregat].kabupaten.nunique()} | tahun={sorted(d.tahun.unique())}')

[INFO] produksi_jenis: 7 kode kembar Kab/Kota dipisahkan (kemunculan pertama = Kabupaten): Blitar, Kediri, Madiun, Malang, Mojokerto, Pasuruan, Probolinggo
[INFO] luas: 7 kode kembar Kab/Kota dipisahkan (kemunculan pertama = Kabupaten): Blitar, Kediri, Madiun, Malang, Mojokerto, Pasuruan, Probolinggo
produksi  baris= 1368 | kab unik=38 | tahun=[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
rtp       baris= 1140 | kab unik=38 | tahun=[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
luas      baris= 1140 | kab unik=38 | tahun=[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


## 5. Audit tahun — produksi per jenis, RTP, luas
Tabel ini menggantikan bukti manual v2 (2A, 2C, 2D, 2E) dan **otomatis berlaku untuk tahun baru**.

In [6]:
# --- produksi opendata: produksi per (kab, jenis, tahun)
pj = prod_od[['tahun', 'kabupaten', 'jenis', 'nilai']].copy()
Wp = pj.groupby(['kabupaten', 'tahun'])['nilai'].sum(min_count=1).unstack()
a_pj = audit_years(Wp); print('PRODUKSI (opendata) - audit tahun:'); display(a_pj.round(3))
pj, _ = apply_audit(pj, 'produksi_jenis')
pj_agg = prod_raw[prod_raw.agregat].groupby('tahun')['nilai'].sum()      # total provinsi bila disediakan

# --- RTP
if rtp_od is not None:
    rt = rtp_od[~rtp_od.agregat][['tahun', 'kabupaten', 'jenis', 'nilai']].rename(columns={'nilai': 'rtp'})
    rt, a_rt = apply_audit(rt.rename(columns={'rtp': 'nilai'}), 'rtp', stale=True); rt = rt.rename(columns={'nilai': 'rtp'})
    print('RTP - audit tahun:'); display(a_rt.round(3))
else: rt = None

PRODUKSI (opendata) - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.000,-0.046,0.072,0.000,38
2021,skala,0.001,2.976,0.039,0.000,38
2022,ok,1.000,-0.016,0.024,0.000,38
2023,ok,1.000,0.012,0.020,0.000,38
2024,tidak_konsisten,1.000,-2.992,1.558,0.711,11
2025,ok,1.000,0.040,0.038,0.000,38


[WARN] produksi_jenis: tahun 2021 berskala x1000 -> dikoreksi
[WARN] produksi_jenis: tahun 2024 tidak_konsisten (zero_share=71%, median_log10=-2.99) -> NaN
[WARN] rtp: tahun 2024 salinan tahun 2022 (>=90% kabupaten identik) -> NaN
RTP - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.0,-0.002,0.048,0.000,38
2021,ok,1.0,-0.007,0.101,0.184,31
2022,ok,1.0,0.000,0.009,0.000,38
2023,ok,1.0,0.002,0.055,0.000,38
2024,salinan,1.0,0.000,0.009,0.000,38


### 5b. Luas lahan — diperbaiki, bukan dibuang
Urutan perbaikan (tiap langkah dicatat):
1. baris yang **persis sama dengan angka RTP** pada kabupaten-jenis-tahun yang sama → itu angka rumah tangga, bukan hektar → `NaN`;
2. **audit tahun** (skala/tidak lengkap/tidak konsisten/salinan) → tahun rusak jadi `NaN`;
3. **lonjakan baris tunggal** (mis. satu baris 23 juta ha) → `NaN` untuk baris itu saja, tahun lain dan baris lain tetap dipakai;
4. bila file **KKP lahan budidaya** tersedia → dipakai sebagai sumber utama, opendata hanya mengisi tahun yang KKP-nya tidak valid *dan* satuannya terbukti sebanding.

In [7]:
def clean_luas_opendata(lu, rt):
    lu = lu[~lu.agregat][['tahun', 'kabupaten', 'jenis', 'nilai']].copy(); n0 = int((lu.nilai > 0).sum())
    if rt is not None:                                                                    # (1) sama persis dengan RTP
        m = lu.merge(rt[['tahun', 'kabupaten', 'jenis', 'rtp']], on=['tahun', 'kabupaten', 'jenis'], how='left')
        eq = ((m.nilai > 0) & (m.nilai == m.rtp)).values
        qc('INFO', 'luas_opendata', f'{int(eq.sum())} baris luas persis sama dengan RTP -> NaN (per tahun: {dict(lu[eq].tahun.value_counts().sort_index())})')
        lu.loc[eq, 'nilai'] = np.nan
    lu, aud = apply_audit(lu, 'luas_opendata', stale=True)                                # (2) audit tahun
    lu, ns = drop_spikes(lu, ['kabupaten', 'jenis'])                                      # (3) lonjakan baris
    if ns: qc('WARN', 'luas_opendata', f'{ns} baris lonjakan >50x median barisnya sendiri -> NaN (tahun lain tetap dipakai)'); log('luas_opendata', 'lonjakan baris tunggal', f'{ns} baris', 'baris -> NaN; tahun & kabupaten lain tetap dipakai')
    return lu, aud

luas_clean = None
if luas_od is not None:
    luas_clean, a_lu = clean_luas_opendata(luas_od, rt)
    print('LUAS (opendata) - audit tahun:'); display(a_lu.round(3))
    tot = luas_clean.groupby('tahun')['nilai'].agg(total_ha='sum', n_baris_valid=lambda s: int(s.notna().sum()), maksimum='max'); print('Setelah perbaikan:'); display(tot.round(0))

[INFO] luas_opendata: 52 baris luas persis sama dengan RTP -> NaN (per tahun: {2021: np.int64(5), 2022: np.int64(47)})
[WARN] luas_opendata: tahun 2024 tidak_konsisten (zero_share=74%, median_log10=-1.09) -> NaN
[WARN] luas_opendata: 5 baris lonjakan >50x median barisnya sendiri -> NaN (tahun lain tetap dipakai)
LUAS (opendata) - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.0,-0.251,1.210,0.000,38
2021,ok,1.0,0.000,0.330,0.000,38
2022,sebagian,1.0,0.086,0.638,0.263,28
2023,ok,1.0,0.020,0.690,0.000,38
2024,tidak_konsisten,1.0,-1.094,1.050,0.737,10


Setelah perbaikan:


,total_ha,n_baris_valid,maksimum
tahun,,,
2020,52248.0,227,15601.0
2021,166944.0,222,40980.0
2022,1229101.0,180,842251.0
2023,230213.0,226,70000.0
2024,0.0,0,NaN


In [8]:
def load_luas_kkp():
    '''File ke-10 (KKP Lahan Budidaya). Struktur belum diketahui pasti -> deteksi kolom otomatis, gagal = dilewati (bukan error).'''
    d = read('luas_kkp')
    if d is None: return None
    try:
        tc = pick(d, r'^tahun'); kc = pick(d, r'kabupaten|kota|wilayah'); pc = pick(d, r'provinsi', required=False)
        jc = pick(d, r'jenis|kategori|kelompok|media|komoditas', required=False)
        vcols = [c for c in d.columns if not str(c).startswith('_') and re.search(r'luas|lahan|hektar|\bha\b', norm_txt(c)) and c not in (kc, jc)]
        if not vcols: raise KeyError('kolom luas tidak terdeteksi')
        o = pd.DataFrame({'tahun': pd.to_numeric(d[tc], errors='coerce'), 'kabupaten': d[kc].map(norm_kab), '_file': d['_file']})
        o['nilai'] = pd.concat([d[c].map(parse_number) for c in vcols], axis=1).sum(axis=1, min_count=1)
        o['jenis'] = d[jc].astype(str).str.upper().str.strip() if jc else 'TOTAL'
        if pc: o = o[d[pc].map(norm_txt).str.contains('jawa timur')]
        o = o.dropna(subset=['tahun', 'kabupaten']); o['tahun'] = o.tahun.astype(int); o = newest_per_year(o)
        o = o[~o.jenis.map(norm_txt).isin(['total', 'jumlah', 'jumlah total'])]
        qc('INFO', 'luas_kkp', f'terbaca: {len(o)} baris, kolom luas={vcols}, kolom jenis={jc}, tahun={sorted(o.tahun.unique())}')
        o, aud = apply_audit(o[['tahun', 'kabupaten', 'jenis', 'nilai']], 'luas_kkp', stale=True)
        o, ns = drop_spikes(o, ['kabupaten', 'jenis']) if o.jenis.nunique() > 0 else (o, 0)
        return o
    except Exception as e:
        qc('WARN', 'luas_kkp', f'file ada tetapi tidak bisa dibaca otomatis ({type(e).__name__}: {e}); dilewati. Cek nama kolom.'); return None

luas_kkp = load_luas_kkp()

[INFO] luas_kkp: terbaca: 50 baris, kolom luas=['Luas Lahan Budi Daya'], kolom jenis=Jenis Usaha, tahun=[np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
[INFO] luas_kkp: tahun 2020 tak terverifikasi (pembanding <2 tahun)
[INFO] luas_kkp: tahun 2021 tak terverifikasi (pembanding <2 tahun)
[INFO] luas_kkp: tahun 2022 tak terverifikasi (pembanding <2 tahun)
[INFO] luas_kkp: tahun 2023 tak terverifikasi (pembanding <2 tahun)
[INFO] luas_kkp: tahun 2024 tak terverifikasi (pembanding <2 tahun)


**Memilih sumber luas per kabupaten-tahun.** KKP diutamakan; opendata hanya mengisi tahun yang KKP-nya kosong/rusak
**bila** rasio median keduanya pada tahun yang sama-sama valid berada di 0,5–2 (satuan sebanding). Bila tidak, tidak dicampur.

In [9]:
def kab_year(d, name):
    if d is None: return None
    return d.groupby(['kabupaten', 'tahun'])['nilai'].sum(min_count=1).rename(name).reset_index()

ly_od, ly_kk = kab_year(luas_clean, 'luas_od'), kab_year(luas_kkp, 'luas_kkp')
if ly_od is None and ly_kk is None:
    qc('WARN', 'luas', 'tidak ada sumber luas -> luas_lahan_ha = NaN untuk semua; dashboard menyembunyikan metrik ini'); fl = pd.DataFrame(columns=['kabupaten', 'tahun', 'luas_lahan_ha', 'luas_sumber'])
else:
    base = ly_od if ly_kk is None else (ly_kk if ly_od is None else ly_kk.merge(ly_od, on=['kabupaten', 'tahun'], how='outer'))
    for c in ('luas_od', 'luas_kkp'):
        if c not in base: base[c] = np.nan
    both = base.dropna(subset=['luas_od', 'luas_kkp']); both = both[(both.luas_od > 0) & (both.luas_kkp > 0)]
    comparable = bool(len(both) >= 5 and 0.5 <= (both.luas_od / both.luas_kkp).median() <= 2)
    if len(both) >= 5: qc('INFO', 'luas', f'rasio median opendata/KKP = {(both.luas_od / both.luas_kkp).median():.2f} (n={len(both)}) -> ' + ('sebanding, opendata boleh mengisi' if comparable else 'TIDAK sebanding, tidak dicampur'))
    use_od = base.luas_od if (comparable or base.luas_kkp.isna().all()) else pd.Series(np.nan, index=base.index)
    base['luas_lahan_ha'] = base.luas_kkp.where(base.luas_kkp.notna(), use_od)
    base['luas_sumber'] = np.where(base.luas_kkp.notna(), 'kkp', np.where(base.luas_lahan_ha.notna(), 'opendata', None))
    fl = base[['kabupaten', 'tahun', 'luas_lahan_ha', 'luas_sumber']]
cov = fl.groupby('tahun').agg(kab_terisi=('luas_lahan_ha', lambda s: int(s.notna().sum())), total_ha=('luas_lahan_ha', 'sum')) if len(fl) else pd.DataFrame()
print('Luas lahan KANONIK (dipakai dashboard):'); display(cov.round(0))
n_ok = int((cov.kab_terisi > 0).sum()) if len(cov) else 0
log('luas', 'luas lahan rusak pada sebagian tahun/baris', f'tahun valid: {n_ok} dari {len(cov)}', 'diperbaiki per baris/tahun; yang rusak -> NaN (bukan dibuang seluruhnya); KKP diutamakan bila ada')

[INFO] luas: rasio median opendata/KKP = 0.00 (n=9) -> TIDAK sebanding, tidak dicampur
Luas lahan KANONIK (dipakai dashboard):


,kab_terisi,total_ha
tahun,,
2020,2,289779.0
2021,1,148084.0
2022,3,688845.0
2023,3,451177.0
2024,3,450780.0


## 6. KKP: pembesaran (kanonik), pembenihan, ikan hias, pembudidaya

In [10]:
def kkp_table(key, names, pats):
    d = read(key)
    if d is None: return None
    cols = [c for c in d.columns if c != '_file']; first = pd.to_numeric(d[cols[0]], errors='coerce')
    if len(cols) == len(names) and first.notna().mean() > .9: d = d.rename(columns=dict(zip(cols, names)))
    else: d = d.rename(columns={pick(d, *pats[n]): n for n in names})
    d['tahun'] = pd.to_numeric(d['tahun'], errors='coerce'); d = d.dropna(subset=['tahun']); d['tahun'] = d.tahun.astype(int)
    n_all = len(d); d = d[d['prov'].map(norm_txt).str.contains('jawa timur')].copy()
    if n_all != len(d): qc('INFO', key, f'{n_all - len(d)} baris non-Jawa Timur dibuang')
    d['kabupaten'] = d['kab'].map(norm_kab)
    for c in names:
        if c.startswith(('vol', 'nilai', 'harga')): d[c] = d[c].map(parse_number)
    return newest_per_year(d)

PAT8 = dict(tahun=[r'^tahun'], prov=[r'provinsi'], kab=[r'kabupaten|kota'], kelompok=[r'kelompok|komoditas'], jenis_ikan=[r'^jenis'], vol_ton=[r'volume|produksi'], nilai_juta=[r'^nilai'], harga_kg=[r'^harga'])
kk = kkp_table('pembesaran', ['tahun', 'prov', 'kab', 'kelompok', 'jenis_ikan', 'vol_ton', 'nilai_juta', 'harga_kg'], PAT8)
if kk is None: raise SystemExit('Berhenti: file produksi pembesaran (KKP) WAJIB ada.')
kk['komoditas'] = kk['kelompok'].astype(str).str.upper().str.strip().replace(SYN_KOM)
kk['grup'] = kk['komoditas'].map(GRUP_KOM).fillna('lainnya')
nd = int(kk.duplicated(['tahun', 'kabupaten', 'komoditas', 'jenis_ikan']).sum())
if nd: qc('WARN', 'pembesaran', f'{nd} baris kunci ganda (tahun,kab,komoditas,jenis) -> ambil yang terakhir'); kk = kk.drop_duplicates(['tahun', 'kabupaten', 'komoditas', 'jenis_ikan'], keep='last')
kk, a_kk = apply_audit(kk.rename(columns={'vol_ton': 'nilai'}), 'pembesaran_kkp'); kk = kk.rename(columns={'nilai': 'vol_ton'})
bad_y = [int(y) for y, r in a_kk.iterrows() if r.status in ('tidak_lengkap', 'tidak_konsisten', 'salinan')]
kk.loc[kk.tahun.isin(bad_y), 'nilai_juta'] = np.nan                                         # nilai ikut NaN bila volumenya tahun rusak
print('PEMBESARAN KKP (kanonik) - audit tahun:'); display(a_kk.round(3))
fp = kk[['tahun', 'kabupaten', 'komoditas', 'grup', 'jenis_ikan', 'vol_ton', 'nilai_juta', 'harga_kg']].copy()

PAT7 = dict(tahun=[r'^tahun'], prov=[r'provinsi'], kab=[r'kabupaten|kota'], kelompok=[r'kelompok|komoditas|jenis'], vol=[r'volume|produksi|jumlah'], nilai_juta=[r'^nilai'], harga=[r'^harga'])
def small(key):
    d = kkp_table(key, ['tahun', 'prov', 'kab', 'kelompok', 'vol', 'nilai_juta', 'harga'], PAT7)
    if d is None: return None
    d, aud = apply_audit(d[['tahun', 'kabupaten', 'kelompok', 'vol', 'nilai_juta', 'harga']].rename(columns={'vol': 'nilai'}), key, lenient=True); d = d.rename(columns={'nilai': 'vol'})
    print(f'{key.upper()} - audit tahun:'); display(aud.round(3)); return d
pb, hs = small('pembenihan'), small('hias')

# --- pembudidaya
pem = read('pembudidaya')
pc_k, pc_t, pc_v = pick(pem, r'kabupaten|kota'), pick(pem, r'^tahun'), pick(pem, r'jumlah', r'pembudidaya')
pv = pd.DataFrame({'kabupaten': pem[pc_k].map(norm_kab), 'tahun': pd.to_numeric(pem[pc_t], errors='coerce'), 'pembudidaya': pem[pc_v].map(parse_number), '_file': pem['_file']}).dropna(subset=['tahun', 'kabupaten'])
pv['tahun'] = pv.tahun.astype(int); pv = newest_per_year(pv); pv = pv[~pv.kabupaten.map(norm_txt).isin(['jawa timur'])].drop_duplicates(['kabupaten', 'tahun'], keep='last')
pv_a, a_pv = apply_audit(pv.rename(columns={'pembudidaya': 'nilai'}), 'pembudidaya', stale=True); pv = pv_a.rename(columns={'nilai': 'pembudidaya'})
print('PEMBUDIDAYA - audit tahun:'); display(a_pv.round(3))
pv['median_kab'] = pv.groupby('kabupaten')['pembudidaya'].transform('median'); pv['rasio'] = pv.pembudidaya / pv.median_kab
pv['flag_lonjakan'] = ((pv.rasio > 3) | (pv.rasio < 1 / 3)) & (pv.groupby('kabupaten')['pembudidaya'].transform('count') >= 3)
print('Pembudidaya: titik >3x / <1/3x median kab-nya sendiri:'); display(pv[pv.flag_lonjakan][['kabupaten', 'tahun', 'pembudidaya', 'median_kab', 'rasio']].round(2))
log('pembudidaya', 'lonjakan >3x median kabupaten sendiri', f'{int(pv.flag_lonjakan.sum())} titik', 'diganti median tahun-tahun lain kab tsb (flag disimpan)')

PEMBESARAN KKP (kanonik) - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.0,-0.038,0.068,0.0,38
2021,ok,1.0,-0.024,0.044,0.0,38
2022,ok,1.0,-0.003,0.032,0.0,38
2023,ok,1.0,0.019,0.042,0.0,38
2024,ok,1.0,0.034,0.039,0.0,38


PEMBENIHAN - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.0,-0.054,0.193,0.029,34
2021,ok,1.0,-0.016,0.081,0.029,34
2022,ok,1.0,0.002,0.085,0.029,34
2023,ok,1.0,0.012,0.051,0.029,34
2024,ok,1.0,0.037,0.082,0.086,32


[INFO] hias: 910 baris non-Jawa Timur dibuang
HIAS - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.0,0.002,0.273,0.148,23
2021,ok,1.0,-0.003,0.116,0.037,26
2022,ok,1.0,0.001,0.086,0.037,26
2023,ok,1.0,0.003,0.165,0.037,26
2024,ok,1.0,0.005,0.441,0.038,26


PEMBUDIDAYA - audit tahun:


,status,faktor,median_log10,iqr_log10,zero_share,n_kab_ada
tahun,,,,,,
2020,ok,1.0,-0.003,0.089,0.0,38
2021,ok,1.0,-0.000,0.046,0.0,38
2022,ok,1.0,0.000,0.023,0.0,38
2023,ok,1.0,0.001,0.021,0.0,38
2024,ok,1.0,0.000,0.031,0.0,38


Pembudidaya: titik >3x / <1/3x median kab-nya sendiri:


,kabupaten,tahun,pembudidaya,median_kab,rasio
17,Nganjuk,2020,10360.0,1465.0,7.07
36,Kota Surabaya,2020,5404.0,1262.0,4.28
38,Pacitan,2021,542.0,1839.0,0.29
74,Kota Surabaya,2021,5404.0,1262.0,4.28
167,Mojokerto,2024,14065.0,1455.0,9.67


## 7. Rekonsiliasi dengan angka resmi BPS (pembanding independen, tidak dijumlahkan)

In [11]:
def bps_totals():
    out = []
    bk = read('bps_kegiatan')
    if bk is not None:
        try:
            cols = [c for c in bk.columns if c != '_file']; d = bk.rename(columns=dict(zip(cols[:5], ['kab', 'pembesaran', 'pembenihan', 'hias', 'tahun'])))
            d['tahun'] = pd.to_numeric(d['tahun'], errors='coerce'); d = d.dropna(subset=['tahun']); d['pembesaran'] = d.pembesaran.map(parse_number)
            t = d[d.kab.map(norm_txt) == 'jawa timur']; out.append(pd.DataFrame({'tahun': t.tahun.astype(int), 'bps_jatim_ton': t.pembesaran, 'sumber': 'bps_kegiatan'}))
            kab_lvl = d[d.kab.map(norm_txt) != 'jawa timur'].assign(kabupaten=lambda x: x.kab.map(norm_kab), tahun=lambda x: x.tahun.astype(int))[['tahun', 'kabupaten', 'pembesaran']]
        except Exception as e: qc('WARN', 'bps_kegiatan', f'tidak terbaca ({e})'); kab_lvl = None
    else: kab_lvl = None
    bn_all = []
    for p in find_files('bps_nilai'):
        try:
            d = read_any(p); d.columns = [str(c).strip() for c in d.columns]
            kc, pc, nc = pick(d, r'kabupaten|kota'), pick(d, r'produksi.*jumlah|jumlah.*produksi|^produksi'), pick(d, r'nilai.*jumlah|jumlah.*nilai|^nilai')
            tc = pick(d, r'^tahun', required=False); d['kab'] = d[kc]; tot = d[kc].map(norm_txt) == 'jawa timur'
            if tc: d['tahun'] = pd.to_numeric(d[tc], errors='coerce')
            else:
                a, b = (int(x) for x in re.findall(r'(20\d\d)', p.stem)[-2:]) if len(re.findall(r'(20\d\d)', p.stem)) >= 2 else (None, None)
                blk = tot.shift(fill_value=False).cumsum(); yrs = list(range(a, b + 1)) if a else []
                if a and blk.nunique() == len(yrs): d['tahun'] = blk.map(dict(zip(sorted(blk.unique()), yrs)))
                else: qc('WARN', 'bps_nilai', f'{p.name}: tidak ada kolom tahun & jumlah blok ({blk.nunique()}) != jumlah tahun di nama file ({len(yrs)}) -> dilewati'); continue
            d['bps_ton'] = d[pc].map(parse_number) / 1000; d['bps_nilai_juta'] = d[nc].map(parse_number) / 1e6; bn_all.append(d.assign(_tot=tot)[['kab', 'tahun', 'bps_ton', 'bps_nilai_juta', '_tot']])
        except Exception as e: qc('WARN', 'bps_nilai', f'{p.name} tidak terbaca ({type(e).__name__}: {e})')
    bn = pd.concat(bn_all) if bn_all else None
    if bn is not None:
        t = bn[bn._tot]; out.append(pd.DataFrame({'tahun': t.tahun.astype(int), 'bps_jatim_ton': t.bps_ton, 'sumber': 'bps_nilai'}))
        bn = bn[~bn._tot].assign(kabupaten=lambda x: x.kab.map(norm_kab), tahun=lambda x: x.tahun.astype(int))[['tahun', 'kabupaten', 'bps_ton', 'bps_nilai_juta']]
    tots = pd.concat(out) if out else pd.DataFrame(columns=['tahun', 'bps_jatim_ton', 'sumber'])
    return tots, kab_lvl, bn
bps_tot, bps_kab_keg, bps_kab_nilai = bps_totals()

kkp_tot = fp.groupby('tahun')['vol_ton'].sum(min_count=1).rename('kkp_ton')
rec = pd.concat([kkp_tot, bps_tot.drop_duplicates('tahun').set_index('tahun')['bps_jatim_ton']], axis=1).dropna(subset=['kkp_ton'])
rec['selisih_%'] = (rec.kkp_ton / rec.bps_jatim_ton - 1) * 100
print('Total KKP vs total resmi BPS Jatim (ton):'); display(rec.round(1))
for y, r in rec.dropna(subset=['selisih_%']).iterrows():
    if abs(r['selisih_%']) > 5: qc('WARN', 'rekonsiliasi', f'total KKP {int(y)} berbeda {r["selisih_%"]:+.1f}% dari BPS Jatim', int(y))

Total KKP vs total resmi BPS Jatim (ton):


,kkp_ton,bps_jatim_ton,selisih_%
tahun,,,
2020,1264158.9,1264159.0,-0.0
2021,1279954.4,1279959.0,-0.0
2022,1284129.9,1284129.0,0.0
2023,1347519.6,495.9,271625.5
2024,1392701.6,696.7,199813.8


[WARN] rekonsiliasi: total KKP 2023 berbeda +271625.5% dari BPS Jatim
[WARN] rekonsiliasi: total KKP 2024 berbeda +199813.8% dari BPS Jatim


## 8. Membangun tabel fakta bersih

In [12]:
# 8a. produksi per jenis budidaya (opendata, terkoreksi); tahun rusak = NaN
fj = pj.rename(columns={'nilai': 'produksi_ton'}).copy(); fj['valid'] = fj.produksi_ton.notna()
fj['skala_dikoreksi'] = fj.tahun.isin([int(y) for y, r in a_pj.iterrows() if r.status == 'skala'])
fj.to_parquet(PROC / 'fact_jenis_budidaya.parquet', index=False)

# 8b. pembesaran KKP kab x komoditas (kanonik)
fp.to_parquet(PROC / 'fact_pembesaran.parquet', index=False)

# 8c. pembudidaya terkoreksi (lonjakan -> median kab dari titik normal)
med_ok = pv[~pv.flag_lonjakan].groupby('kabupaten')['pembudidaya'].median()
pv['pembudidaya_clean'] = np.where(pv.flag_lonjakan, pv.kabupaten.map(med_ok), pv.pembudidaya)
pv[['tahun', 'kabupaten', 'pembudidaya', 'pembudidaya_clean', 'flag_lonjakan']].to_parquet(PROC / 'fact_pembudidaya.parquet', index=False)

# 8d. RTP per kab-tahun (tahun salinan/rusak sudah NaN)
fr = (rt.groupby(['kabupaten', 'tahun'])['rtp'].sum(min_count=1).reset_index() if rt is not None else pd.DataFrame(columns=['kabupaten', 'tahun', 'rtp']))
fr.to_parquet(PROC / 'fact_rtp.parquet', index=False)

# 8e. luas lahan kanonik (kab-tahun) + rincian jenis opendata
fl.to_parquet(PROC / 'fact_luas.parquet', index=False)
(luas_clean.rename(columns={'nilai': 'luas_ha'}) if luas_clean is not None else pd.DataFrame(columns=['tahun', 'kabupaten', 'jenis', 'luas_ha'])).to_parquet(PROC / 'fact_luas_jenis.parquet', index=False)

# 8f. pembenihan & ikan hias (volume saja; tahun rusak NaN)
(pb[['tahun', 'kabupaten', 'kelompok', 'vol']].rename(columns={'vol': 'vol_ribu_ekor'}) if pb is not None else pd.DataFrame(columns=['tahun', 'kabupaten', 'kelompok', 'vol_ribu_ekor'])).to_parquet(PROC / 'fact_pembenihan.parquet', index=False)
(hs[['tahun', 'kabupaten', 'kelompok', 'vol']].rename(columns={'vol': 'vol_ekor'}) if hs is not None else pd.DataFrame(columns=['tahun', 'kabupaten', 'kelompok', 'vol_ekor'])).to_parquet(PROC / 'fact_ikanhias.parquet', index=False)

# 8g. produksi opendata per kab-tahun untuk tahun YANG TIDAK ADA di KKP (mis. 2025) -> validasi forecast
kk_years = set(fp.tahun[fp.vol_ton.notna()].unique())
ext = fj[~fj.tahun.isin(kk_years)].groupby(['kabupaten', 'tahun'])['produksi_ton'].sum(min_count=1).reset_index().dropna()
ext = ext[ext.produksi_ton > 0]; ext.to_parquet(PROC / 'validasi_produksi_tahun_baru.parquet', index=False)

# 8h. pembanding BPS (untuk fact_budidaya di notebook 05)
parts = []
if bps_kab_nilai is not None: parts += [bps_kab_nilai.rename(columns={'bps_ton': 'produksi_ton', 'bps_nilai_juta': 'nilai_rp_juta'}).melt(['tahun', 'kabupaten'], var_name='metrik', value_name='nilai').assign(sumber='bps_nilai')]
if bps_kab_keg is not None: parts += [bps_kab_keg.rename(columns={'pembesaran': 'nilai'}).assign(metrik='produksi_ton', sumber='bps_kegiatan')]
(pd.concat(parts) if parts else pd.DataFrame(columns=['tahun', 'kabupaten', 'metrik', 'nilai', 'sumber'])).to_parquet(PROC / 'fact_bps.parquet', index=False)
bps_tot.to_parquet(PROC / 'bps_totals.parquet', index=False)
print('Tabel fakta tersimpan:', sorted(p.name for p in PROC.glob('*.parquet')))

Tabel fakta tersimpan: ['bps_totals.parquet', 'fact_bps.parquet', 'fact_ikanhias.parquet', 'fact_jenis_budidaya.parquet', 'fact_luas.parquet', 'fact_luas_jenis.parquet', 'fact_pembenihan.parquet', 'fact_pembesaran.parquet', 'fact_pembudidaya.parquet', 'fact_rtp.parquet', 'validasi_produksi_tahun_baru.parquet']


## 9. Laporan: audit tahun, QC, kebijakan, dan jejak data

In [13]:
ya = pd.concat(YEAR_AUDIT, ignore_index=True) if YEAR_AUDIT else pd.DataFrame(); ya.to_csv(REP / 'year_audit.csv', index=False)
q = pd.DataFrame(QC); q.to_csv(REP / 'qc_report.csv', index=False)
pd.DataFrame(LOG).to_csv(REP / 'data_quality_log.csv', index=False)
kebijakan = pd.DataFrame([
 ['Produksi/nilai pembesaran (KKP)', 'audit tahun generik', 'skala dikoreksi; tahun tak lengkap/tak konsisten -> NaN', 'baris tak ada = komoditas tak dibudidayakan (0 struktural)'],
 ['Produksi per jenis (opendata)', 'nol = belum lapor, skala, kab/kota kembar', 'koreksi skala / NaN; kembar dipisah generik', 'total provinsi dari KKP'],
 ['Pembudidaya', 'lonjakan >3x median kab', 'diganti median kab sendiri', 'kabupaten tetap ikut'],
 ['RTP', 'salinan antar tahun', 'tahun salinan -> NaN', 'dipakai bila cakupan cukup'],
 ['Luas lahan', 'campur data RTP / lonjakan / tahun nol', 'diperbaiki per baris & tahun; KKP diutamakan', 'tahun rusak -> NaN, TIDAK dibuang seluruhnya'],
 ['Pembenihan / ikan hias', 'skala & tahun rusak', 'tahun rusak -> NaN; ringkas dgn median antar tahun valid', 'kabupaten tak tercatat = 0 struktural']], columns=['variabel', 'kondisi', 'perlakuan', 'catatan'])
kebijakan.to_csv(REP / 'kebijakan_missing.csv', index=False)
files = {k: [dict(nama=p.name, ukuran=p.stat().st_size, mtime=int(p.stat().st_mtime)) for p in find_files(k)] for k in RULES}
data_hash = hashlib.sha1(json.dumps(files, sort_keys=True).encode()).hexdigest()[:12]
json.dump({'files': files, 'data_hash': data_hash, 'tahun_kkp': sorted(int(y) for y in kk_years), 'tahun_semua': sorted(int(y) for y in set(fp.tahun) | set(pj.tahun))}, open(REP / 'run_meta.json', 'w'), indent=1)
print(f'{len(LOG)} keputusan cleaning | QC: {q.level.value_counts().to_dict() if len(q) else {}} | data_hash={data_hash}')
display(ya[ya.status != 'ok'][['dataset', 'tahun', 'status', 'faktor', 'zero_share', 'median_log10']].round(3))
if len(q) and (q.level == 'ERROR').any(): display(q[q.level == 'ERROR'])

6 keputusan cleaning | QC: {'INFO': 11, 'WARN': 7} | data_hash=a9938b8fc259


,dataset,tahun,status,faktor,zero_share,median_log10
1,produksi_jenis,2021,skala,0.001,0.000,2.976
4,produksi_jenis,2024,tidak_konsisten,1.000,0.711,-2.992
10,rtp,2024,salinan,1.000,0.000,0.000
13,luas_opendata,2022,sebagian,1.000,0.263,0.086
15,luas_opendata,2024,tidak_konsisten,1.000,0.737,-1.094
16,luas_kkp,2020,tak_terverifikasi,1.000,NaN,NaN
17,luas_kkp,2021,tak_terverifikasi,1.000,NaN,NaN
18,luas_kkp,2022,tak_terverifikasi,1.000,NaN,NaN
19,luas_kkp,2023,tak_terverifikasi,1.000,NaN,NaN
20,luas_kkp,2024,tak_terverifikasi,1.000,NaN,NaN
